# 01 — Classes et attributs

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- définir une classe avec `class`
- écrire un constructeur `__init__` et comprendre `self`
- distinguer attributs d'instance et attributs de classe
- définir des méthodes, les appeler, les chaîner
- maîtriser `__str__`/`__repr__` pour l'affichage et le debug
- utiliser le contexte métier du fil rouge (réservation de salles)

## Prérequis — ce que vous connaissez déjà

À ce stade de la formation intermédiaire, vous maîtrisez :

- toute l'Initiation 5j : types, fonctions typées, modules, fichiers, CSV/JSON
- les f-strings (`f"{x:.2f}"`, `f"{x=}"`)
- `def`, type hints modernes (`int | None`, `list[T]`)
- exceptions `try`/`except`
- `@dataclass` à voir plus tard — ici on définit tout à la main pour comprendre

Ce que nous n'avons **pas encore vu** (et que nous n'utiliserons donc pas dans ce notebook) :

- `@property`, `@classmethod`, `@staticmethod` (notebook 02)
- l'héritage et `super()` (notebook 03)
- la surcharge d'opérateurs au-delà de `__str__`/`__repr__` (notebook 04)
- `ABC`, `Protocol`, typage abstrait (notebook 05)
- `@dataclass` (section 03 — on en parle très brièvement en synthèse)

## Plan

1. De la fonction à la classe : pourquoi
2. Première classe : `Salle`
3. Le constructeur `__init__` et `self`
4. Attributs d'instance vs attributs de classe
5. Méthodes : opérations sur l'objet
6. Affichage : `__str__` et `__repr__`
7. Mutabilité et identité
8. Encapsulation par convention : `_` et `__`
9. Pattern `from_*` : constructeurs alternatifs *(preview)*
10. Synthèse
11. Exercices

---

## 1. De la fonction à la classe : pourquoi

Jusqu'ici vous manipulez des données avec des **fonctions** et des **dicts**. Ça fonctionne, mais lorsque la même structure doit être créée, validée, affichée, comparée à plein d'endroits, les fonctions s'empilent sans cohérence. Une **classe** regroupe sous un seul nom :

- la **structure** (quels attributs) ;
- le **comportement** (quelles méthodes) ;
- les **invariants** (ce qui doit rester vrai).

### L'approche « dict + fonctions » — ce qu'on avait avant

Voici le pattern le plus courant en Initiation : un dictionnaire par entité, des fonctions qui prennent ce dictionnaire en premier argument.

In [ ]:
def creer_salle(nom: str, capacite: int) -> dict:
    return {"nom": nom, "capacite": capacite}


In [ ]:
salle = creer_salle("Mars", 12)


In [ ]:
salle


Et pour décrire la salle, une fonction libre :

In [ ]:
def decrire_salle(s: dict) -> str:
    return f"Salle {s['nom']} ({s['capacite']} places)"


In [ ]:
decrire_salle(salle)


Ça marche, mais :

- rien n'empêche un autre code d'écrire `salle["nomm"] = "X"` (faute de frappe silencieuse) ;
- la capacité pourrait devenir `-12` sans que personne ne s'en plaigne ;
- pour trouver **tout** ce qui concerne une salle, il faut chercher dans le code.

---

## 2. Première classe : `Salle`

Une classe se déclare avec le mot-clé `class`. Par convention, son nom est en **PascalCase**.

In [ ]:
class Salle:
    pass


On crée une **instance** en appelant la classe comme une fonction :

In [ ]:
salle_vide = Salle()


In [ ]:
type(salle_vide)


In [ ]:
isinstance(salle_vide, Salle)


À ce stade la classe est vide : elle ne contient aucune donnée, aucune méthode. Elle sert juste à créer des objets d'un type bien à elle. Regardons comment lui attacher des données à la création.

---

## 3. Le constructeur `__init__` et `self`

La méthode `__init__` est appelée **automatiquement** à chaque création d'instance. C'est elle qui attache les attributs. Son premier paramètre, par convention nommé `self`, est l'instance en cours de construction.

In [ ]:
class Salle:
    def __init__(self, nom: str, capacite: int) -> None:
        self.nom = nom
        self.capacite = capacite


In [ ]:
mars = Salle("Mars", 12)


In [ ]:
mars.nom


In [ ]:
mars.capacite


### Décortiquer l'appel

Quand on écrit `Salle("Mars", 12)`, Python :

1. crée un nouvel objet vide de type `Salle` ;
2. appelle `Salle.__init__(nouvel_objet, "Mars", 12)` — c'est pour ça que `self` est le **premier** paramètre mais qu'on ne le passe **jamais** à la main ;
3. retourne l'objet initialisé.

Retenez : **`self` n'est pas un mot-clé**, c'est juste un nom conventionnel. On pourrait écrire `this`, `moi` ou n'importe quoi, mais ne le faites pas — tous les développeurs Python attendent `self`.

### Plusieurs instances sont indépendantes

In [ ]:
venus = Salle("Venus", 6)


In [ ]:
venus.nom, mars.nom


In [ ]:
venus.capacite, mars.capacite


Chaque instance a son propre « sac d'attributs ». Modifier l'une ne touche pas l'autre.

In [ ]:
venus.capacite = 8


In [ ]:
venus.capacite, mars.capacite


---

## 4. Attributs d'instance vs attributs de classe

Un attribut défini **dans le corps de la classe**, en dehors de toute méthode, appartient à la **classe elle-même**. Il est partagé par toutes les instances.

In [ ]:
class Salle:
    # attribut de classe — partagé
    organisation = "Acme Corp"

    def __init__(self, nom: str, capacite: int) -> None:
        # attributs d'instance — propres à chaque objet
        self.nom = nom
        self.capacite = capacite


In [ ]:
a = Salle("A", 4)


In [ ]:
b = Salle("B", 8)


In [ ]:
a.organisation


In [ ]:
b.organisation


In [ ]:
Salle.organisation


Modifier l'attribut **via la classe** change la valeur vue par toutes les instances qui n'ont pas réaffecté leur propre attribut du même nom :

In [ ]:
Salle.organisation = "Acme Europe"


In [ ]:
a.organisation, b.organisation


### ⚠️ Piège : attribut de classe mutable

Ne déclarez **jamais** une liste ou un dict comme attribut de classe destiné à être modifié par instance. Tous les objets partageraient la même liste.

In [ ]:
class SalleMauvaise:
    reservations: list[str] = []  # ← piège classique

    def __init__(self, nom: str) -> None:
        self.nom = nom


In [ ]:
x = SalleMauvaise("X")
y = SalleMauvaise("Y")
x.reservations.append("lundi 9h")
y.reservations  # contient "lundi 9h" — horreur


La règle est simple : **tout état mutable se crée dans `__init__`**, via `self.xxx = ...`.

---

## 5. Méthodes : opérations sur l'objet

Une méthode est une fonction définie **dans** la classe. Son premier paramètre est toujours `self`. On y accède sur l'instance.

In [ ]:
class Salle:
    def __init__(self, nom: str, capacite: int) -> None:
        self.nom = nom
        self.capacite = capacite
        self.reservations: list[str] = []

    def reserver(self, creneau: str) -> None:
        self.reservations.append(creneau)

    def est_libre(self, creneau: str) -> bool:
        return creneau not in self.reservations


In [ ]:
mars = Salle("Mars", 12)


In [ ]:
mars.reserver("lundi 9h")


In [ ]:
mars.reservations


In [ ]:
mars.est_libre("lundi 9h")


In [ ]:
mars.est_libre("mardi 14h")


### Méthode et appel par la classe

Écrire `mars.reserver(...)` est équivalent à `Salle.reserver(mars, ...)`. La première forme est celle qu'on utilise partout ; la seconde est utile à connaître pour lire des erreurs.

In [ ]:
Salle.reserver(mars, "mardi 10h")


In [ ]:
mars.reservations


---

## 6. Affichage : `__str__` et `__repr__`

Sans configuration, l'affichage d'une instance est peu parlant :

In [ ]:
mars


In [ ]:
print(mars)


### `__repr__` — destinée aux développeurs

`__repr__` doit renvoyer une chaîne **sans ambiguïté** qui permet d'identifier l'objet. L'idéal : un texte qui, si on le recopie dans un REPL, reconstruit l'objet (ou un équivalent).

In [ ]:
class Salle:
    def __init__(self, nom: str, capacite: int) -> None:
        self.nom = nom
        self.capacite = capacite

    def __repr__(self) -> str:
        return f"Salle(nom={self.nom!r}, capacite={self.capacite})"


In [ ]:
Salle("Mars", 12)


### `__str__` — destinée aux utilisateurs

`__str__` est appelée par `str()`, `print()`, et les f-strings. Si elle n'existe pas, Python retombe sur `__repr__`.

In [ ]:
class Salle:
    def __init__(self, nom: str, capacite: int) -> None:
        self.nom = nom
        self.capacite = capacite

    def __repr__(self) -> str:
        return f"Salle(nom={self.nom!r}, capacite={self.capacite})"

    def __str__(self) -> str:
        return f"Salle {self.nom} ({self.capacite} places)"


In [ ]:
mars = Salle("Mars", 12)


In [ ]:
mars  # appelle __repr__


In [ ]:
print(mars)  # appelle __str__


In [ ]:
f"Réservation confirmée dans la {mars}"


---

## 7. Mutabilité et identité

Une instance est **mutable par défaut** : on peut changer ses attributs après construction. C'est pratique mais dangereux.

In [ ]:
mars = Salle("Mars", 12)
mars.capacite = 20  # mutation
mars


Deux instances différentes sont égales uniquement si **une règle explicite** est définie. Par défaut, `==` se rabat sur l'identité (`is`), c'est-à-dire l'adresse mémoire :

In [ ]:
a = Salle("Mars", 12)
b = Salle("Mars", 12)
a == b  # False par défaut !


In [ ]:
a is b


On verra au notebook **04 — Surcharge d'opérateurs** comment définir `__eq__` pour obtenir une égalité structurelle.

---

## 8. Encapsulation par convention : `_` et `__`

Python n'a **pas** de mot-clé `private`. L'encapsulation est conventionnelle :

- un nom préfixé d'un underscore (`_foo`) signifie « détail interne, ne pas utiliser de l'extérieur » ; Python ne vous empêchera rien, c'est un contrat social ;
- un nom préfixé de deux underscores (`__foo`) déclenche le **name mangling** : l'attribut devient `_NomClasse__foo`. C'est surtout utile pour éviter les collisions lors de l'héritage (notebook 03).

In [ ]:
class Salle:
    def __init__(self, nom: str, capacite: int) -> None:
        self.nom = nom
        self._capacite = capacite  # attribut interne

    def capacite(self) -> int:
        return self._capacite


In [ ]:
mars = Salle("Mars", 12)


In [ ]:
mars.capacite()


In [ ]:
mars._capacite  # ← on peut techniquement y accéder, mais on ne devrait pas


Dans le notebook **02 — Propriétés**, on verra `@property` qui permet d'écrire `mars.capacite` (sans parenthèses) tout en gardant un contrôle interne.

---

## 9. Pattern `from_*` : constructeurs alternatifs *(preview)*

Souvent on veut créer un objet à partir d'un dict, d'une ligne CSV, d'un JSON. La convention Pythonique est de nommer ces constructeurs `from_xxx`. On peut les écrire comme des méthodes de classe dès maintenant, sans `@classmethod`, en utilisant une simple **méthode statique par convention** via `@staticmethod` (que nous verrons proprement au notebook 02).

Pour l'instant, on l'écrit comme une **fonction libre** à côté de la classe :

In [ ]:
class Salle:
    def __init__(self, nom: str, capacite: int) -> None:
        self.nom = nom
        self.capacite = capacite

    def __repr__(self) -> str:
        return f"Salle({self.nom!r}, {self.capacite})"


def salle_depuis_dict(data: dict[str, object]) -> Salle:
    """Construit une Salle à partir d'un dictionnaire (ex. ligne JSON)."""
    return Salle(nom=str(data["nom"]), capacite=int(data["capacite"]))


In [ ]:
salle_depuis_dict({"nom": "Terre", "capacite": 20})


---

## Synthèse

| Élément | Rôle | Exemple |
|---|---|---|
| `class Nom:` | Déclarer une classe | `class Salle:` |
| `__init__(self, ...)` | Constructeur, attache les attributs | `self.nom = nom` |
| `self` | Instance courante, 1er paramètre de toute méthode | `self.reserver(...)` |
| Attribut d'instance | Propre à chaque objet, défini dans `__init__` | `self.capacite = 12` |
| Attribut de classe | Partagé, défini dans le corps de la classe | `organisation = "Acme"` |
| Méthode | Fonction de la classe, 1er paramètre `self` | `def reserver(self, c): ...` |
| `__repr__` | Affichage **développeur**, sans ambiguïté | `Salle('Mars', 12)` |
| `__str__` | Affichage **utilisateur**, humain | `Salle Mars (12 places)` |


### Règles à retenir

1. **Tout état mutable se crée dans `__init__`**, jamais en attribut de classe.
2. **`self` est un nom conventionnel** ; il est toujours le premier paramètre des méthodes d'instance.
3. **PascalCase** pour le nom de classe, **snake_case** pour méthodes et attributs.
4. **Implémenter `__repr__` au minimum** sur toute classe métier : ça sauve des heures de debug.
5. **Ne pas confondre `is` et `==`** : par défaut `==` compare l'identité, pas le contenu (voir notebook 04).
6. **Pas de `private` en Python** : `_attr` est un contrat social, `__attr` déclenche le name mangling.

---

## Exercices

Tous les exercices sont à rédiger sous forme de **fonctions ou méthodes typées** (PEP 604, type hints modernes). Les docstrings sont obligatoires sur les méthodes publiques.

### Exercice 1 — Classe `Utilisateur` *(facile)*

Écrire une classe `Utilisateur` avec :

- deux attributs : `nom: str` et `email: str` ;
- un `__repr__` de la forme `Utilisateur(nom='Alice', email='alice@ex.fr')` ;
- un `__str__` de la forme `Alice <alice@ex.fr>`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Classes_et_attributs", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
class Utilisateur:
    def __init__(self, nom: str, email: str) -> None:
        self.nom = nom
        self.email = email

    def __repr__(self) -> str:
        return f"Utilisateur(nom={self.nom!r}, email={self.email!r})"

    def __str__(self) -> str:
        return f"{self.nom} <{self.email}>"


alice = Utilisateur("Alice", "alice@ex.fr")
print(repr(alice))
print(alice)
```

</details>

### Exercice 2 — Compteur avec attribut de classe *(facile)*

Écrire une classe `Compteur` qui compte, à travers un attribut de **classe** `total`, le nombre d'instances créées. Chaque appel à `__init__` doit incrémenter ce compteur. Vérifier qu'après avoir créé 3 `Compteur()`, `Compteur.total` vaut `3`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Classes_et_attributs", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
class Compteur:
    total: int = 0

    def __init__(self) -> None:
        Compteur.total += 1


a = Compteur()
b = Compteur()
c = Compteur()
print(Compteur.total)  # 3
```

</details>

### Exercice 3 — Méthode qui modifie l'état *(facile)*

Écrire une classe `Panier` avec :

- un attribut `articles: list[str]` initialisé vide dans `__init__` ;
- une méthode `ajouter(self, article: str) -> None` ;
- une méthode `total(self) -> int` qui renvoie le nombre d'articles ;
- un `__str__` qui affiche `Panier(N articles)`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Classes_et_attributs", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
class Panier:
    def __init__(self) -> None:
        self.articles: list[str] = []

    def ajouter(self, article: str) -> None:
        self.articles.append(article)

    def total(self) -> int:
        return len(self.articles)

    def __str__(self) -> str:
        return f"Panier({self.total()} articles)"


p = Panier()
p.ajouter("pain")
p.ajouter("lait")
print(p)  # Panier(2 articles)
```

</details>

### Exercice 4 — Classe `Reservation` (fil rouge) *(moyen)*

Écrire une classe `Reservation` avec :

- attributs : `salle: str`, `creneau: str`, `organisateur: str` ;
- un attribut **de classe** `suivantes: int = 1` qui sert à attribuer un `id` auto-incrémenté ;
- dans `__init__`, affecter `self.id` puis incrémenter `Reservation.suivantes` ;
- `__repr__` : `Reservation(id=1, salle='Mars', creneau='lundi 9h', organisateur='Alice')` ;
- `__str__` : `#1 · Mars · lundi 9h · par Alice`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Classes_et_attributs", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
class Reservation:
    suivantes: int = 1

    def __init__(self, salle: str, creneau: str, organisateur: str) -> None:
        self.id = Reservation.suivantes
        Reservation.suivantes += 1
        self.salle = salle
        self.creneau = creneau
        self.organisateur = organisateur

    def __repr__(self) -> str:
        return (
            f"Reservation(id={self.id}, salle={self.salle!r}, "
            f"creneau={self.creneau!r}, organisateur={self.organisateur!r})"
        )

    def __str__(self) -> str:
        return f"#{self.id} · {self.salle} · {self.creneau} · par {self.organisateur}"


r1 = Reservation("Mars", "lundi 9h", "Alice")
r2 = Reservation("Venus", "mardi 14h", "Bob")
print(r1)
print(r2)
```

</details>

### Exercice 5 — Salle avec validation dans `__init__` *(moyen)*

Écrire une classe `Salle` qui **valide** ses paramètres dans `__init__` :

- `nom` doit être une chaîne non vide, sinon lever `ValueError('nom vide')` ;
- `capacite` doit être un entier strictement positif, sinon `ValueError('capacité invalide')`.

Tester que `Salle('Mars', 12)` fonctionne, que `Salle('', 12)` et `Salle('X', 0)` lèvent bien l'exception attendue.

**Indice :** utiliser `try`/`except` pour vérifier le comportement.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Classes_et_attributs", exercice=5)


<details>
<summary>📖 Voir la correction</summary>

```python
class Salle:
    def __init__(self, nom: str, capacite: int) -> None:
        if not nom:
            raise ValueError("nom vide")
        if capacite <= 0:
            raise ValueError("capacité invalide")
        self.nom = nom
        self.capacite = capacite

    def __repr__(self) -> str:
        return f"Salle({self.nom!r}, {self.capacite})"


print(Salle("Mars", 12))

for nom, cap in [("", 12), ("X", 0), ("Y", -3)]:
    try:
        Salle(nom, cap)
    except ValueError as exc:
        print(f"refusé : {exc}")
```

</details>

### Exercice 6 — Transfert d'état entre instances *(moyen)*

Écrire une classe `Compte` représentant un compte bancaire avec :

- attributs : `titulaire: str`, `solde: float` ;
- méthode `deposer(self, montant: float) -> None` ;
- méthode `retirer(self, montant: float) -> None` qui lève `ValueError` si le solde serait négatif ;
- méthode `virement(self, autre: 'Compte', montant: float) -> None` qui retire `montant` de `self` et le dépose sur `autre`.

Vérifier : après `a.virement(b, 30)`, les deux soldes ont bien changé.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Classes_et_attributs", exercice=6)


<details>
<summary>📖 Voir la correction</summary>

```python
class Compte:
    def __init__(self, titulaire: str, solde: float = 0.0) -> None:
        self.titulaire = titulaire
        self.solde = solde

    def deposer(self, montant: float) -> None:
        self.solde += montant

    def retirer(self, montant: float) -> None:
        if montant > self.solde:
            raise ValueError("solde insuffisant")
        self.solde -= montant

    def virement(self, autre: "Compte", montant: float) -> None:
        self.retirer(montant)
        autre.deposer(montant)

    def __repr__(self) -> str:
        return f"Compte({self.titulaire!r}, solde={self.solde:.2f})"


a = Compte("Alice", 100.0)
b = Compte("Bob", 50.0)
a.virement(b, 30.0)
print(a, b)
```

</details>

### Exercice 7 — Mini-gestionnaire de réservations *(difficile)*

Écrire deux classes reliées :

- `Salle` : `nom`, `capacite`, méthode `__repr__` ;
- `Agenda` : contient une liste `reservations: list[tuple[Salle, str]]` ; expose `reserver(self, salle: Salle, creneau: str) -> None` qui ajoute la paire si elle n'existe pas déjà (sinon lève `ValueError('déjà réservé')`), `creneaux_de(self, salle: Salle) -> list[str]`, et `__str__` qui affiche une ligne par réservation.

Tester le tout sur deux salles et trois créneaux.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Classes_et_attributs", exercice=7)


<details>
<summary>📖 Voir la correction</summary>

```python
class Salle:
    def __init__(self, nom: str, capacite: int) -> None:
        self.nom = nom
        self.capacite = capacite

    def __repr__(self) -> str:
        return f"Salle({self.nom!r}, {self.capacite})"


class Agenda:
    def __init__(self) -> None:
        self.reservations: list[tuple[Salle, str]] = []

    def reserver(self, salle: Salle, creneau: str) -> None:
        if (salle, creneau) in self.reservations:
            raise ValueError("déjà réservé")
        self.reservations.append((salle, creneau))

    def creneaux_de(self, salle: Salle) -> list[str]:
        return [c for (s, c) in self.reservations if s is salle]

    def __str__(self) -> str:
        lignes = [f"{s.nom} — {c}" for (s, c) in self.reservations]
        return "\n".join(lignes) or "(aucune réservation)"


mars = Salle("Mars", 12)
venus = Salle("Venus", 6)
ag = Agenda()
ag.reserver(mars, "lundi 9h")
ag.reserver(venus, "lundi 9h")
ag.reserver(mars, "mardi 14h")
print(ag)
print(ag.creneaux_de(mars))
```

</details>

### Exercice 8 — Vector 2D — premières briques de surcharge *(difficile)*

Écrire une classe `Vec2` représentant un vecteur 2D :

- attributs `x: float`, `y: float` ;
- méthode `norme(self) -> float` qui renvoie la norme euclidienne (utiliser `math.hypot`) ;
- méthode `plus(self, autre: 'Vec2') -> 'Vec2'` qui renvoie un **nouveau** `Vec2` somme (ne pas modifier `self`) ;
- `__repr__` de la forme `Vec2(3, 4)`.

Note : la surcharge de `+` sera vue au notebook 04. Ici on passe par une méthode nommée.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Classes_et_attributs", exercice=8)


<details>
<summary>📖 Voir la correction</summary>

```python
import math


class Vec2:
    def __init__(self, x: float, y: float) -> None:
        self.x = x
        self.y = y

    def norme(self) -> float:
        return math.hypot(self.x, self.y)

    def plus(self, autre: "Vec2") -> "Vec2":
        return Vec2(self.x + autre.x, self.y + autre.y)

    def __repr__(self) -> str:
        return f"Vec2({self.x}, {self.y})"


u = Vec2(3, 4)
v = Vec2(1, 2)
print(u.norme())  # 5.0
print(u.plus(v))  # Vec2(4, 6)
```

</details>

---

## Ressources externes

### Documentation officielle
- [Classes — tutoriel officiel](https://docs.python.org/3/tutorial/classes.html)
- [Data model — `__init__`, `__repr__`, `__str__`](https://docs.python.org/3/reference/datamodel.html)

### PEPs de référence
- **PEP 8** — [Naming conventions](https://peps.python.org/pep-0008/#naming-conventions) : `CapWords` pour les classes
- **PEP 257** — [Docstring conventions](https://peps.python.org/pep-0257/)

### Lectures complémentaires
- Raymond Hettinger — *Beyond PEP 8* (talk PyCon 2015) : l'idiome Python au-delà de la syntaxe.
- Fluent Python (Ramalho), chap. 9 *A Pythonic Object* : `__repr__`, `__str__` et l'état interne.